# Capstone Project — Study Buddy (Physics)
**Agentic AI Course 2026 | Dr. Kanthi Kiran Sirra**

---
**Student Problem Statement**

- **Domain:** Physics — B.Tech Engineering syllabus
- **User:** B.Tech students studying Physics at odd hours without access to tutors
- **Problem:** Students struggle to understand core physics concepts (mechanics, optics, thermodynamics, electromagnetism, etc.) from textbooks alone. They need an intelligent assistant that can explain topics clearly, faithfully, and without hallucinating formulas or values.
- **Success:** The agent correctly explains concepts from the KB, cites source topics, admits when a question is out of scope, and remembers the student's name and prior questions within a session.
- **Tool:** `datetime` tool (to wish good morning/evening based on time) + `calculator` tool (to evaluate numeric physics expressions like F=ma given values).
---

## Part 1 — Domain Setup: Knowledge Base

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install langgraph langchain langchain-groq chromadb sentence-transformers streamlit ragas

In [ ]:
import os
from getpass import getpass

# Enter your own Groq API key at runtime.
# Do NOT write the actual key in this notebook before submitting.
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("Groq API key loaded for this notebook session only.")


In [ ]:
# ── Knowledge Base: 12 Physics Documents ─────────────────────────────────────
DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "Newton's Laws of Motion",
        "text": (
            "Newton's First Law (Law of Inertia): An object at rest stays at rest, and an object in motion "
            "stays in motion with the same speed and direction unless acted upon by an unbalanced external force. "
            "Newton's Second Law: The net force acting on an object equals the product of its mass and acceleration: "
            "F = ma, where F is force in Newtons (N), m is mass in kilograms (kg), and a is acceleration in m/s². "
            "Newton's Third Law: For every action, there is an equal and opposite reaction. "
            "Example: When you push against a wall with 10 N, the wall pushes back on you with 10 N in the opposite direction. "
            "Applications: These laws explain why seatbelts are necessary (inertia), how rockets propel in space (action-reaction), "
            "and how brakes decelerate vehicles (net force changes velocity). "
            "Important: Newton's laws apply in inertial (non-accelerating) reference frames."
        )
    },
    {
        "id": "doc_002",
        "topic": "Work, Energy and Power",
        "text": (
            "Work is defined as the product of force and displacement in the direction of force: W = F × d × cos(θ), "
            "where θ is the angle between force and displacement. Unit: Joule (J). "
            "Kinetic Energy (KE) is the energy of motion: KE = ½mv², where m = mass (kg) and v = velocity (m/s). "
            "Potential Energy (PE) is stored energy due to position: PE = mgh, where g = 9.8 m/s² and h = height (m). "
            "Law of Conservation of Energy: Energy cannot be created or destroyed; it only transforms. "
            "Total mechanical energy E = KE + PE = constant (in absence of friction). "
            "Power is the rate of doing work: P = W/t, Unit: Watt (W). Also P = F × v. "
            "Work-Energy Theorem: The net work done on an object equals its change in kinetic energy: W_net = ΔKE."
        )
    },
    {
        "id": "doc_003",
        "topic": "Gravitation",
        "text": (
            "Newton's Law of Universal Gravitation: Every two masses attract each other with a force: "
            "F = G × m1 × m2 / r², where G = 6.674 × 10⁻¹¹ N·m²/kg² (universal gravitational constant), "
            "m1 and m2 are the masses, and r is the distance between their centers. "
            "Acceleration due to gravity on Earth's surface: g = 9.8 m/s². "
            "Escape velocity: The minimum velocity to escape Earth's gravitational field is v_escape = √(2GM/R) ≈ 11.2 km/s. "
            "Orbital velocity: v_orbital = √(GM/r). "
            "Kepler's Laws: (1) Planets orbit in ellipses with the sun at one focus. "
            "(2) Equal areas are swept in equal times. "
            "(3) T² ∝ r³ — the square of the orbital period is proportional to the cube of the semi-major axis."
        )
    },
    {
        "id": "doc_004",
        "topic": "Thermodynamics",
        "text": (
            "Thermodynamics deals with heat, temperature, and energy transfer. "
            "Zeroth Law: If body A is in thermal equilibrium with B, and B with C, then A and C are also in equilibrium. "
            "First Law (Conservation of Energy): ΔU = Q − W, where ΔU = change in internal energy, Q = heat added to system, W = work done by system. "
            "Second Law: Heat flows spontaneously from hot to cold. The entropy of an isolated system never decreases. "
            "Third Law: As temperature approaches absolute zero (0 K), entropy approaches a minimum constant value. "
            "Specific Heat Capacity (c): Q = mcΔT. For water, c = 4200 J/kg·K. "
            "Ideal Gas Law: PV = nRT, where P = pressure (Pa), V = volume (m³), n = moles, R = 8.314 J/mol·K, T = temperature (K). "
            "Isothermal: T constant. Adiabatic: Q = 0. Isochoric: V constant. Isobaric: P constant."
        )
    },
    {
        "id": "doc_005",
        "topic": "Waves and Oscillations",
        "text": (
            "Simple Harmonic Motion (SHM): Restoring force ∝ displacement: F = −kx. "
            "Period of a spring-mass system: T = 2π√(m/k). Period of a simple pendulum: T = 2π√(L/g). "
            "Waves are periodic disturbances that carry energy. Transverse waves: displacement ⊥ to propagation (e.g., light). "
            "Longitudinal waves: displacement ∥ to propagation (e.g., sound). "
            "Wave equation: v = fλ, where v = wave speed (m/s), f = frequency (Hz), λ = wavelength (m). "
            "Speed of sound in air at 0°C ≈ 331 m/s; increases ~0.6 m/s per °C. "
            "Doppler Effect: Apparent frequency changes when source or observer moves. "
            "f_observed = f_source × (v ± v_observer) / (v ∓ v_source). "
            "Resonance: When driving frequency equals natural frequency, amplitude is maximum."
        )
    },
    {
        "id": "doc_006",
        "topic": "Optics — Reflection and Refraction",
        "text": (
            "Law of Reflection: Angle of incidence = Angle of reflection (both measured from the normal). "
            "Refraction: When light passes from one medium to another, it bends. Snell's Law: n1 sin(θ1) = n2 sin(θ2). "
            "Refractive index n = c / v, where c = speed of light in vacuum = 3 × 10⁸ m/s, v = speed in medium. "
            "Total Internal Reflection occurs when light travels from denser to rarer medium and angle of incidence > critical angle. "
            "Critical angle θ_c = sin⁻¹(n2/n1). Applications: optical fibers, diamonds. "
            "Mirror formula: 1/f = 1/v + 1/u (sign convention: distances measured from pole). "
            "Lens formula: 1/f = 1/v − 1/u. Magnification m = v/u. "
            "Convex lens: converging, real images (mostly). Concave lens: diverging, always virtual images."
        )
    },
    {
        "id": "doc_007",
        "topic": "Electrostatics",
        "text": (
            "Coulomb's Law: Force between two point charges: F = k × q1 × q2 / r², "
            "where k = 9 × 10⁹ N·m²/C² (Coulomb's constant), q1 and q2 are charges (Coulombs), r = distance (m). "
            "Electric Field E = F/q₀ = kQ/r² (N/C or V/m). Field lines go from + to −. "
            "Electric Potential V = kQ/r (Volts). Potential is a scalar. "
            "Relation: E = −dV/dr. "
            "Capacitance C = Q/V (Farads). Parallel plate capacitor: C = ε₀A/d. "
            "ε₀ = 8.85 × 10⁻¹² F/m (permittivity of free space). "
            "Gauss's Law: Total electric flux through a closed surface = Q_enclosed / ε₀. "
            "Energy stored in capacitor: U = ½CV² = Q²/2C."
        )
    },
    {
        "id": "doc_008",
        "topic": "Current Electricity",
        "text": (
            "Electric current I = Q/t (Amperes). Conventional current flows from + to −. "
            "Ohm's Law: V = IR, where V = voltage (V), I = current (A), R = resistance (Ω). "
            "Resistance R = ρL/A, where ρ = resistivity (Ω·m), L = length (m), A = cross-section area (m²). "
            "Series: R_total = R1 + R2 + ... Current same through all. "
            "Parallel: 1/R_total = 1/R1 + 1/R2 + ... Voltage same across all. "
            "Power: P = VI = I²R = V²/R (Watts). "
            "Kirchhoff's Laws: (1) KCL — Sum of currents at a junction = 0. (2) KVL — Sum of voltages around a loop = 0. "
            "EMF (ε) and internal resistance (r): Terminal voltage V = ε − Ir. "
            "Wheatstone bridge: balanced when R1/R2 = R3/R4 — used to find unknown resistance."
        )
    },
    {
        "id": "doc_009",
        "topic": "Magnetism and Electromagnetic Induction",
        "text": (
            "Magnetic force on a moving charge: F = qvB sin(θ), where B = magnetic field (Tesla). "
            "Force on a current-carrying conductor: F = BIL sin(θ). "
            "Biot-Savart Law: dB = μ₀/4π × I dl × r̂ / r². "
            "Ampere's Law: ∮B·dl = μ₀I_enclosed. "
            "μ₀ = 4π × 10⁻⁷ T·m/A (permeability of free space). "
            "Magnetic field at center of circular loop: B = μ₀I / 2R. "
            "Faraday's Law of Induction: EMF = −dΦ/dt, where Φ = magnetic flux = B·A·cos(θ). "
            "Lenz's Law: Induced current opposes the change that caused it. "
            "Transformer: V1/V2 = N1/N2. Step-up: N2 > N1. Step-down: N2 < N1. "
            "Self-inductance L: EMF = −L dI/dt. Energy stored: U = ½LI²."
        )
    },
    {
        "id": "doc_010",
        "topic": "Modern Physics — Photoelectric Effect and Atomic Models",
        "text": (
            "Photoelectric Effect (Einstein, 1905): Light strikes a metal surface and ejects electrons. "
            "Energy of photon: E = hf = hc/λ, where h = 6.626 × 10⁻³⁴ J·s (Planck's constant). "
            "KE_max = hf − φ, where φ = work function (minimum energy to eject electron). "
            "Threshold frequency: f₀ = φ/h. Below f₀, no emission regardless of intensity. "
            "Bohr's Atomic Model: Electrons orbit nucleus in fixed energy levels. "
            "Energy of nth level in hydrogen: Eₙ = −13.6/n² eV. Ground state: n=1, E = −13.6 eV. "
            "When electron transitions from n2 → n1: ΔE = hf = 13.6(1/n1² − 1/n2²) eV. "
            "de Broglie wavelength: λ = h/mv — matter has wave properties. "
            "Heisenberg Uncertainty Principle: Δx · Δp ≥ h/4π."
        )
    },
    {
        "id": "doc_011",
        "topic": "Projectile Motion and Kinematics",
        "text": (
            "Kinematics equations (uniform acceleration): "
            "(1) v = u + at, (2) s = ut + ½at², (3) v² = u² + 2as, (4) s = (u+v)/2 × t. "
            "Projectile Motion: Horizontal and vertical components are independent. "
            "Horizontal: x = u_x × t = u cos(θ) × t (constant velocity, no acceleration). "
            "Vertical: y = u sin(θ) × t − ½gt² (free fall under gravity). "
            "Time of flight: T = 2u sin(θ)/g. "
            "Maximum height: H = u²sin²(θ)/2g. "
            "Horizontal range: R = u²sin(2θ)/g — maximum range at θ = 45°. "
            "Relative velocity: v_AB = v_A − v_B. Used in river-crossing and rain problems."
        )
    },
    {
        "id": "doc_012",
        "topic": "Rotational Motion",
        "text": (
            "Angular displacement (θ), angular velocity (ω = dθ/dt), angular acceleration (α = dω/dt). "
            "Kinematic analogies: θ = ω₀t + ½αt², ω² = ω₀² + 2αθ. "
            "Torque: τ = r × F = Iα, where I = moment of inertia. "
            "Moment of inertia depends on mass distribution: I = Σmr². "
            "For a solid disk: I = ½MR². For a solid sphere: I = 2/5 MR². For a rod (about center): I = ML²/12. "
            "Parallel axis theorem: I = I_cm + Md². "
            "Angular momentum: L = Iω. Conservation: if τ_net = 0, L = constant. "
            "Rolling without slipping: v_cm = Rω. Total KE = ½Mv² + ½Iω². "
            "Centripetal acceleration: a_c = v²/r = ω²r. Centripetal force: F_c = mv²/r."
        )
    }
]

print(f"Knowledge base loaded: {len(DOCUMENTS)} documents")
for doc in DOCUMENTS:
    print(f"  {doc['id']} — {doc['topic']}")

In [ ]:
# ── Build ChromaDB Vector Store ───────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import chromadb

print("Loading SentenceTransformer...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Create in-memory ChromaDB collection
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="physics_kb")

# Add all documents
doc_texts  = [d["text"]  for d in DOCUMENTS]
doc_ids    = [d["id"]    for d in DOCUMENTS]
doc_metas  = [{"topic": d["topic"]} for d in DOCUMENTS]

print("Embedding documents...")
embeddings = embedder.encode(doc_texts).tolist()

collection.add(
    documents=doc_texts,
    embeddings=embeddings,
    ids=doc_ids,
    metadatas=doc_metas
)
print(f"✅ ChromaDB populated with {collection.count()} documents.")

In [ ]:
# ── Retrieval Test (MUST pass before building graph) ─────────────────────────
test_queries = [
    "What is Newton's second law?",
    "How does total internal reflection work?",
    "Explain Faraday's law of induction",
    "What is the photoelectric effect?",
    "How do I calculate projectile range?"
]

print("=== Retrieval Test ===")
for q in test_queries:
    q_emb = embedder.encode([q]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=2)
    topics = [m["topic"] for m in results["metadatas"][0]]
    print(f"Q: {q}")
    print(f"   → Retrieved: {topics}")
print("✅ Retrieval test complete — verify topics are relevant before proceeding.")

## Part 2 — State Design

In [ ]:
from typing import TypedDict, List, Optional

class CapstoneState(TypedDict):
    # Core fields (mandatory)
    question:      str
    messages:      List[dict]          # full conversation history
    route:         str                 # 'retrieve' | 'tool' | 'memory_only'
    retrieved:     str                 # formatted context string from ChromaDB
    sources:       List[str]           # list of topic names retrieved
    tool_result:   str                 # result from tool node
    answer:        str                 # LLM-generated answer
    faithfulness:  float               # RAGAS-style self-eval score 0.0-1.0
    eval_retries:  int                 # number of eval retries so far
    # Domain-specific fields
    student_name:  Optional[str]       # extracted from 'my name is ...'

print("✅ CapstoneState defined with fields:")
print(list(CapstoneState.__annotations__.keys()))

## Part 3 — Node Functions

In [ ]:
# ── LLM Setup ────────────────────────────────────────────────────────────────
from langchain_groq import ChatGroq

# Uses GROQ_API_KEY from the current environment/session.
# The key is entered at runtime and is not saved permanently in the notebook.
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2
)
print("✅ Groq LLM ready.")


In [ ]:
# ── Node 1: memory_node ───────────────────────────────────────────────────────
def memory_node(state: CapstoneState) -> dict:
    """Append question to history, apply sliding window, extract student name."""
    msgs = state.get("messages", [])
    question = state["question"]

    # Append current question to history
    msgs = msgs + [{"role": "user", "content": question}]

    # Sliding window — keep last 6 messages to prevent token overflow
    msgs = msgs[-6:]

    # Extract student name if introduced
    student_name = state.get("student_name", None)
    lower_q = question.lower()
    if "my name is" in lower_q:
        parts = lower_q.split("my name is")[-1].strip().split()
        if parts:
            student_name = parts[0].capitalize()

    return {
        "messages": msgs,
        "student_name": student_name
    }

# ── Test memory_node ──────────────────────────────────────────────────────────
test_state = {"question": "Hi, my name is Arjun. What is Newton's second law?",
              "messages": [], "student_name": None}
result = memory_node(test_state)
print("memory_node test:", result)
print("✅ memory_node OK")

In [ ]:
# ── Node 2: router_node ───────────────────────────────────────────────────────
ROUTER_PROMPT = """You are a routing assistant for a Physics Study Buddy.
Given the student's question, decide which route to take:

- retrieve   → The question asks about a physics concept, formula, law, or topic
              that would be in a textbook. Examples: "What is Ohm's law?", "Explain thermodynamics"
- tool       → The question requires calculating a numeric result using given values
              OR asks for the current time/date. Examples: "What is F if m=5 and a=3?", "What time is it?"
- memory_only → Greetings, small talk, or questions about the student's own name/previous context.
              Examples: "Hello", "What did I ask before?", "Do you remember my name?"

Reply with ONE WORD ONLY: retrieve, tool, or memory_only

Student question: {question}"""

def router_node(state: CapstoneState) -> dict:
    """LLM decides route: retrieve | tool | memory_only."""
    prompt = ROUTER_PROMPT.format(question=state["question"])
    response = llm.invoke(prompt)
    route = response.content.strip().lower().split()[0]
    if route not in ["retrieve", "tool", "memory_only"]:
        route = "retrieve"  # safe default
    print(f"  [router] route={route}")
    return {"route": route, "eval_retries": state.get("eval_retries", 0)}

# ── Test router_node ──────────────────────────────────────────────────────────
for q in ["What is Faraday's law?", "Calculate F if m=2kg and a=5 m/s2", "Hello!"]:
    r = router_node({"question": q, "eval_retries": 0})
    print(f"  Q: {q!r:50s} → {r['route']}")
print("✅ router_node OK")

In [ ]:
# ── Node 3: retrieval_node ────────────────────────────────────────────────────
def retrieval_node(state: CapstoneState) -> dict:
    """Embed question, query ChromaDB for top 3 chunks, format as context string."""
    question = state["question"]
    q_emb = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)

    docs    = results["documents"][0]
    metas   = results["metadatas"][0]
    sources = [m["topic"] for m in metas]

    context_parts = [f"[{meta['topic']}]\n{doc}" for doc, meta in zip(docs, metas)]
    context = "\n\n".join(context_parts)

    print(f"  [retrieval] sources={sources}")
    return {"retrieved": context, "sources": sources}

# ── Test retrieval_node ───────────────────────────────────────────────────────
r = retrieval_node({"question": "What is the mirror formula in optics?"})
print(f"  Sources: {r['sources']}")
print(f"  Context snippet: {r['retrieved'][:200]}...")
print("✅ retrieval_node OK")

In [ ]:
# ── Node 4: skip_retrieval_node ───────────────────────────────────────────────
def skip_retrieval_node(state: CapstoneState) -> dict:
    """Used for memory_only route — no KB retrieval needed."""
    return {"retrieved": "", "sources": []}

print("✅ skip_retrieval_node defined")

In [ ]:
# ── Node 5: tool_node ─────────────────────────────────────────────────────────
import datetime
import math

def tool_node(state: CapstoneState) -> dict:
    """
    Handles two tools:
      1. datetime — current date and time
      2. calculator — evaluate safe numeric physics expressions
    Tools NEVER raise exceptions — always return a string.
    """
    question = state["question"].lower()

    # --- Tool 1: Date/Time -------------------------------------------------------
    if any(word in question for word in ["time", "date", "today", "day"]):
        now = datetime.datetime.now()
        result = (f"Current date and time: {now.strftime('%A, %d %B %Y, %I:%M %p')}. "
                  f"Time zone: local system time.")
        print(f"  [tool] datetime → {result}")
        return {"tool_result": result, "retrieved": "", "sources": []}

    # --- Tool 2: Physics Calculator ---------------------------------------------
    # Ask LLM to extract a safe Python math expression
    calc_prompt = (
        f"Extract a single-line Python math expression from this physics question. "
        f"Use only: +, -, *, /, **, math.sqrt(), math.pi, math.sin(), math.cos(). "
        f"Return ONLY the expression, no other text.\n\nQuestion: {state['question']}"
    )
    try:
        expr_response = llm.invoke(calc_prompt)
        expr = expr_response.content.strip().replace("```", "").strip()
        # Safe evaluation
        allowed = {"__builtins__": {}, "math": math}
        value = eval(expr, allowed)
        result = f"Calculation result: {expr} = {value:.4g}"
        print(f"  [tool] calculator → {result}")
    except Exception as e:
        result = f"Calculator error: Could not evaluate expression. Error: {str(e)}"
        print(f"  [tool] calculator error: {e}")

    return {"tool_result": result, "retrieved": "", "sources": []}

# ── Test tool_node ────────────────────────────────────────────────────────────
r1 = tool_node({"question": "What is today's date?"})
print(r1["tool_result"])
r2 = tool_node({"question": "Calculate F if mass=5 kg and acceleration=3 m/s2"})
print(r2["tool_result"])
print("✅ tool_node OK")

In [ ]:
# ── Node 6: answer_node ───────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are Study Buddy, an intelligent physics tutor for B.Tech students.

RULES:
1. Answer ONLY using the provided context. Do NOT add information not in the context.
2. If the context does not contain the answer, say clearly: 
   "I don't have information on that topic in my knowledge base. Please check your textbook or ask your professor."
3. Never fabricate formulas, values, or equations.
4. If a student name is known, address them by name.
5. Keep answers structured: explanation → formula (if any) → example (if relevant).
6. If this is a retry (eval_retries > 0), be even more careful to ground your answer strictly in the context.
"""

def answer_node(state: CapstoneState) -> dict:
    """Build system prompt + context + history, generate answer via LLM."""
    student_name = state.get("student_name", "")
    name_str = f"Student name: {student_name}." if student_name else ""
    eval_retries = state.get("eval_retries", 0)

    # Build context section
    context_section = ""
    if state.get("retrieved"):
        context_section = f"\n\n--- RETRIEVED CONTEXT ---\n{state['retrieved']}"
    if state.get("tool_result"):
        context_section += f"\n\n--- TOOL RESULT ---\n{state['tool_result']}"

    if eval_retries > 0:
        retry_instr = f"\n(This is retry #{eval_retries}. Be strictly grounded in the context above.)"
    else:
        retry_instr = ""

    # Build history string
    history_msgs = state.get("messages", [])[:-1]  # exclude current question
    history_str = ""
    if history_msgs:
        history_str = "\n--- CONVERSATION HISTORY ---\n"
        for m in history_msgs:
            role = m["role"].upper()
            history_str += f"{role}: {m['content']}\n"

    full_prompt = (
        f"{SYSTEM_PROMPT}\n{name_str}"
        f"{history_str}"
        f"{context_section}"
        f"{retry_instr}"
        f"\n\n--- STUDENT QUESTION ---\n{state['question']}"
    )

    response = llm.invoke(full_prompt)
    answer = response.content.strip()
    print(f"  [answer] generated ({len(answer)} chars)")
    return {"answer": answer}

print("✅ answer_node defined")

In [ ]:
# ── Node 7: eval_node ─────────────────────────────────────────────────────────
MAX_EVAL_RETRIES = 2

EVAL_PROMPT = """Rate how faithful the ANSWER is to the CONTEXT below.
Score 0.0 (completely hallucinated) to 1.0 (perfectly grounded in context).
Reply with a single float number only — no text.

CONTEXT:
{context}

ANSWER:
{answer}

SCORE:"""

def eval_node(state: CapstoneState) -> dict:
    """Rate faithfulness; increment retries. Skip if no context retrieved."""
    if not state.get("retrieved") and not state.get("tool_result"):
        # Memory-only — no grounding check needed
        print("  [eval] skipped (memory-only route) → faithfulness=1.0")
        return {"faithfulness": 1.0, "eval_retries": state.get("eval_retries", 0)}

    context = state.get("retrieved", "") or state.get("tool_result", "")
    prompt  = EVAL_PROMPT.format(context=context[:2000], answer=state["answer"])

    try:
        response = llm.invoke(prompt)
        score_str = response.content.strip().split()[0]
        score = float(score_str)
        score = max(0.0, min(1.0, score))
    except Exception:
        score = 0.5  # fallback if parse fails

    retries = state.get("eval_retries", 0)
    gate = "PASS" if score >= 0.7 else "RETRY"
    print(f"  [eval] faithfulness={score:.2f} → {gate} (retries so far: {retries})")

    if score < 0.7:
        retries += 1

    return {"faithfulness": score, "eval_retries": retries}

print("✅ eval_node defined")

In [ ]:
# ── Node 8: save_node ─────────────────────────────────────────────────────────
def save_node(state: CapstoneState) -> dict:
    """Append assistant's answer to messages history."""
    msgs = list(state.get("messages", []))
    msgs.append({"role": "assistant", "content": state["answer"]})
    print("  [save] answer appended to history.")
    return {"messages": msgs}

print("✅ save_node defined")

## Part 4 — Graph Assembly

In [ ]:
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver

# ── Conditional routing functions ─────────────────────────────────────────────
def route_decision(state: CapstoneState) -> str:
    route = state.get("route", "retrieve")
    if route == "tool":
        return "tool"
    elif route == "memory_only":
        return "skip"
    else:
        return "retrieve"

def eval_decision(state: CapstoneState) -> str:
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score < 0.7 and retries <= MAX_EVAL_RETRIES:
        print(f"  [eval_decision] → RETRY (score={score:.2f}, retries={retries})")
        return "answer"  # retry answer_node
    else:
        return "save"

# ── Build Graph ───────────────────────────────────────────────────────────────
graph = StateGraph(CapstoneState)

# Add nodes
graph.add_node("memory",   memory_node)
graph.add_node("router",   router_node)
graph.add_node("retrieve", retrieval_node)
graph.add_node("skip",     skip_retrieval_node)
graph.add_node("tool",     tool_node)
graph.add_node("answer",   answer_node)
graph.add_node("eval",     eval_node)
graph.add_node("save",     save_node)

# Fixed edges
graph.set_entry_point("memory")
graph.add_edge("memory",   "router")
graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")
graph.add_edge("answer",   "eval")
graph.add_edge("save",     "__end__")

# Conditional edges
graph.add_conditional_edges(
    "router",
    route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)
graph.add_conditional_edges(
    "eval",
    eval_decision,
    {"answer": "answer", "save": "save"}
)

# Compile with MemorySaver
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)
print("✅ Graph compiled successfully!")

## Part 5 — Testing

In [ ]:
# ── ask() helper ──────────────────────────────────────────────────────────────
def ask(question: str, thread_id: str = "test-thread") -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    initial_state = {
        "question":     question,
        "messages":     [],
        "route":        "",
        "retrieved":    "",
        "sources":      [],
        "tool_result":  "",
        "answer":       "",
        "faithfulness": 0.0,
        "eval_retries": 0,
        "student_name": None
    }
    result = app.invoke(initial_state, config=config)
    return result

print("✅ ask() helper ready")

In [ ]:
# ── 10 Test Questions ─────────────────────────────────────────────────────────
TEST_QUESTIONS = [
    # Domain questions
    ("Q1 - Newton's 2nd Law",       "Explain Newton's second law of motion with formula."),
    ("Q2 - Optics",                 "What is total internal reflection and where is it used?"),
    ("Q3 - Thermodynamics",         "What does the first law of thermodynamics state?"),
    ("Q4 - Waves",                  "What is the Doppler effect and its formula?"),
    ("Q5 - Electrostatics",         "State Coulomb's law and give the formula."),
    ("Q6 - Magnetism",              "What is Faraday's law of electromagnetic induction?"),
    ("Q7 - Modern Physics",         "Explain the photoelectric effect."),
    ("Q8 - Kinematics",             "What is the formula for horizontal range in projectile motion?"),
    # Red-team questions
    ("Q9 - Out of scope",           "Who invented the internet?"),                        # must admit it doesn't know
    ("Q10 - False premise",         "Is it true that Newton's third law only applies on Earth?"),  # must correct
]

print("=" * 70)
for label, question in TEST_QUESTIONS:
    print(f"\n{'='*70}")
    print(f"{label}: {question}")
    result = ask(question, thread_id="test-main")
    print(f"  Route:       {result.get('route')}")
    print(f"  Sources:     {result.get('sources')}")
    print(f"  Faithfulness:{result.get('faithfulness'):.2f}")
    print(f"  Answer:\n{result.get('answer')[:400]}...")
    print()

In [ ]:
# ── Memory Test: 3-turn conversation ─────────────────────────────────────────
print("=== MEMORY TEST ===")
THREAD = "memory-test-001"

r1 = ask("Hi, my name is Priya. What is Newton's second law?", thread_id=THREAD)
print("Turn 1 answer snippet:", r1["answer"][:200])

r2 = ask("Can you give me an example of it?", thread_id=THREAD)
print("\nTurn 2 answer snippet:", r2["answer"][:200])

r3 = ask("What is my name?", thread_id=THREAD)
print("\nTurn 3 (memory check):", r3["answer"])
print("\n✅ Memory test: The agent should address Priya by name in Turn 3.")

## Part 6 — RAGAS Baseline Evaluation

In [ ]:
# ── 5 QA pairs with ground truth ─────────────────────────────────────────────
EVAL_PAIRS = [
    {
        "question": "What is the formula for kinetic energy?",
        "ground_truth": "Kinetic energy KE = ½mv², where m is mass in kg and v is velocity in m/s."
    },
    {
        "question": "State Newton's law of universal gravitation.",
        "ground_truth": "F = G × m1 × m2 / r², where G = 6.674 × 10⁻¹¹ N·m²/kg²."
    },
    {
        "question": "What is Snell's law?",
        "ground_truth": "n1 sin(θ1) = n2 sin(θ2), relating angles of incidence and refraction."
    },
    {
        "question": "What is the energy stored in a capacitor?",
        "ground_truth": "U = ½CV² = Q²/2C, where C is capacitance in Farads."
    },
    {
        "question": "What is the period of a simple pendulum?",
        "ground_truth": "T = 2π√(L/g), where L is the length of the pendulum and g = 9.8 m/s²."
    }
]

# Collect agent outputs for RAGAS
ragas_data = []
print("Collecting agent responses for RAGAS evaluation...")
for pair in EVAL_PAIRS:
    result = ask(pair["question"], thread_id="ragas-eval")
    ragas_data.append({
        "question":    pair["question"],
        "answer":      result["answer"],
        "contexts":    [result["retrieved"]],
        "ground_truth": pair["ground_truth"]
    })
    print(f"  ✅ {pair['question'][:50]}")
print("Data collection complete.")

In [ ]:
# ── RAGAS / Manual Evaluation ────────────────────────────────────────────────
# RAGAS may use OpenAI internally and require OPENAI_API_KEY.
# If RAGAS/OpenAI credentials are unavailable, run manual faithfulness scoring instead.

import os

def manual_faithfulness_score(context: str, answer: str) -> float:
    context_words = set(context.lower().split())
    answer_words = [word for word in answer.lower().split() if len(word) > 3]
    if not answer_words:
        return 0.0
    overlap = sum(1 for word in answer_words if word in context_words)
    return round(min(1.0, overlap / len(answer_words)), 2)

USE_RAGAS = bool(os.environ.get("OPENAI_API_KEY"))

if USE_RAGAS:
    try:
        from ragas import evaluate
        from ragas.metrics import faithfulness, answer_relevancy, context_precision
        from datasets import Dataset

        dataset = Dataset.from_list(ragas_data)
        scores = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision])
        print("\n=== RAGAS BASELINE SCORES ===")
        print(scores)
    except Exception as exc:
        print(f"RAGAS could not run, using manual fallback instead. Reason: {exc}")
        USE_RAGAS = False

if not USE_RAGAS:
    print("Running manual faithfulness scoring.")
    manual_scores = []
    for item in ragas_data:
        score = manual_faithfulness_score(item["contexts"][0], item["answer"])
        manual_scores.append(score)
        print(f"  Q: {item['question'][:50]:50s} | Manual Faithfulness: {score:.2f}")
    avg = sum(manual_scores) / len(manual_scores) if manual_scores else 0.0
    print(f"\n  Average Manual Faithfulness: {avg:.2f}")


## Part 7 — Deployment (Streamlit)

The runnable product files for this capstone are included in the same folder as this notebook:

- `agent.py` — reusable Study Buddy agent implementation using LangGraph, ChromaDB, SentenceTransformer, Groq, MemorySaver, tools, and eval node
- `capstone_streamlit.py` — Streamlit chat interface

Run the app from the project folder:

```bash
streamlit run capstone_streamlit.py
```

The Streamlit app asks for a Groq API key in the sidebar using a password input. The key is used only for the current running session and is not saved in code, notebook output, GitHub, or project files. The instructor can enter their own Groq key when evaluating.


In [ ]:
# Product-file verification: use the same agent.py that powers Streamlit
import os
from agent import StudyBuddyAgent, DOCUMENTS

product_agent = StudyBuddyAgent(groq_api_key=os.environ["GROQ_API_KEY"])
thread_id = "notebook-product-check"

PRODUCT_TESTS = [
    "Hi, my name is Priya. What is Newton's second law?",
    "What is my name?",
    "What is total internal reflection?",
    "Who invented the internet?",
    "Calculate force if m=5 and a=3",
]

print(f"KB documents: {len(DOCUMENTS)}")
for question in PRODUCT_TESTS:
    result = product_agent.ask(question, thread_id)
    print("\nQ:", question)
    print("Route:", result["route"])
    print("Sources:", result["sources"])
    print("Faithfulness:", result["faithfulness"])
    print("Answer:", result["answer"][:350])


## Part 8 — Written Summary

**Domain:** Physics — B.Tech Engineering Syllabus

**User:** B.Tech students who need 24/7 physics concept help without access to tutors

**What the agent does:**  
Study Buddy is an agentic RAG-based physics tutor. It uses LangGraph to route each student question to retrieval, tool use, or memory-only handling. It retrieves from a 12-document ChromaDB physics knowledge base embedded with SentenceTransformer, answers with Groq, remembers the student's name and recent conversation through MemorySaver + `thread_id`, and checks answer faithfulness before saving the response.

**KB Size:** 12 documents covering Newton's laws, work-energy, gravitation, thermodynamics, waves, optics, electrostatics, current electricity, magnetism, modern physics, kinematics, and rotational motion.

**Tools:** `datetime` for current local date/time, plus a safe AST-based `calculator` for numeric physics expressions such as `F = ma`.

**Deployment:** Streamlit UI in `capstone_streamlit.py`, backed by reusable agent logic in `agent.py`. The UI accepts a Groq API key at runtime through a password field. No API key is hardcoded or submitted.

**Evaluation:** The graph includes a self-reflection faithfulness node. RAGAS can be run when the evaluator provides the required external evaluation credentials; otherwise the notebook includes manual baseline scoring for reporting.

**Product Verification Results:**

| # | Question | Route | Expected Result |
|---|----------|-------|-----------------|
| Q1 | Newton's 2nd Law | retrieve | Grounded physics answer with source |
| Q2 | Total internal reflection | retrieve | Grounded optics answer with source |
| Q3 | Remember student name | memory_only | Recalls session name |
| Q4 | Who invented the internet? | retrieve | Admits out of scope |
| Q5 | Calculate force if m=5 and a=3 | tool | Returns 15 |

**One thing I would improve with more time:**  
I would add a quiz node after the answer node. After explaining a concept, Study Buddy would generate one or two multiple-choice questions from the retrieved source, evaluate the student's answer, and store weak topics in session memory for revision.
